# Recreate Figure 6A 

Steps identified

1. Define biological parameters
2. Initialize weight matrices for connections from HD to CONJ/ENV cells
   1. CONJ cells: random/uniform distribution
   2. ENV cells: zero as it received no input from HD
3. Simulation: for each time step do
   1. calculate firing rate of CONJ cell
   2. calculate firing rate of hd cell at previous time step
   3. update weights of CONJ cell
   4. normalize weights


----


## RSC HD Cells: Model & Equations

1. A ring of head direction (HD) cells, where each cell \( i \) has a **preferred firing direction** \( x_i \) distributed between **0° and 360°**. 
2. Do not excite each other. 
3. Their activity forms a **Gaussian packet**.


## **Mathematical Model of RSC HD Cells**
Each HD cell follows a **rate-coded leaky-integrator neuronal model**, described by the equation:


$ \tau \frac{dA_i}{dt} = -A_i + \sum_j w_{ij} r_j - g \sum_j A_j $

where:
- ( $A_i$ ) is the **activation level** of HD cell ($ i $) at time ($ t $).
- ( $\tau$) is the **time constant**, determining the rate of decay.
- The term ($ -A_i $) ensures **exponential decay**, so in the absence of input, activity decreases over time.
- The term ( $g \sum _j A_j $) represents **global inhibition**, where ($ g $) is a constant that approximates **inhibitory interneuron influence**.


#### **External Input from CONJ and ENV Cells**
Presynaptic input from **CONJ and ENV cells** affects the activation of HD cells through weighted connections:

$I_i = \sum_j w_{ij}^{\text{CONJ}} r_j^{\text{CONJ}} + \sum_j w_{ij}^{\text{ENV}} r_j^{\text{ENV}}$

where:
- ($ w_{ij}^{\text{CONJ}} $) and ($ w_{ij}^{\text{ENV}} $) are the **weights** connecting presynaptic CONJ/ENV cells to HD cell ($ i $).
- ($ r_j^{\text{CONJ}} $) and ($ r_j^{\text{ENV}} $) are the **firing rates** of presynaptic CONJ and ENV cells.
- This input is **summed over all presynaptic cells** and **scaled** by the number of connections each HD cell receives.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

Biological parameters

In [ ]:
# Simulation parameters
N = 100                    # Number of HD cells
theta = np.linspace(0, 360, N, endpoint=False)  # Preferred directions (0-360°)
dt = 0.01                  # Time step (s)
tau = 0.5                  # Time constant for decay (s)
global_inhibition = 0.1     # Strength of inhibitory feedback
num_steps = 500            # Number of simulation steps
sigma = 15                 # Initial Gaussian width (degrees)
input_strength = 0.5        # Strength of external input

Initial activity

In [ ]:
initial_center = 180
activity = norm.pdf(theta, loc=initial_center, scale=sigma)
activity /= np.max(activity)  # Normalize to [0,1]

plt.plot(activity)

Initial connectivity weights

In [ ]:
W_CONJ = np.random.rand(N, N) * 0.1  # Weights from CONJ cells
W_ENV = np.random.rand(N, N) * 0.1   # Weights from ENV cells

Simulation

In [ ]:
history = []

for t in range(num_steps):
    
    # Step 1: Compute inhibition (global inhibitory feedback)
    inhibition = global_inhibition * np.sum(activity) / N  # Normalize inhibition across all cells
    
    # Step 2: Compute presynaptic input from CONJ and ENV cells
    conj_input = W_CONJ @ activity  # Sum of weighted inputs from CONJ cells
    env_input = W_ENV @ activity    # Sum of weighted inputs from ENV cells
    input_current = input_strength * (conj_input + env_input)  # Scale input strength
    
    # Step 3: Apply leaky-integrator update
    d_activity = (-activity + input_current - inhibition) / tau  # Change in activity
    activity += d_activity * dt  # Euler update
    
    # Step 4: Ensure activity remains within valid range [0,1]
    activity = np.clip(activity, 0, 1)
    
    # Step 5: Store activity for visualization
    history.append(activity.copy())

Plotting

In [ ]:
# Convert history to numpy array for plotting
history = np.array(history)

# Plot results
plt.figure(figsize=(10, 5))
plt.imshow(history.T, aspect='auto', cmap='hot', extent=[0, num_steps*dt, 0, 360])
plt.xlabel("Time (s)")
plt.ylabel("Head Direction (°)")
plt.title("HD Cell Activity Over Time")
plt.colorbar(label="Activity Level")

Update `W_CONJ` and `W_ENV` inside the simulation

In [ ]:
# code